# Conversion DWG → DXF → DataFrame → Export

1. **Cellule 1** : Convertit un DWG en DXF (LibreDWG) → variable `dxf_path`
2. **Cellule 2** : Lit le DXF et produit le DataFrame `dxf` (entités, attributs DXF)
3. **Cellules 5–6** : Normalisation des layers (alternatives → layers_geo)
4. **Cellule 7** : Export `export(df)` ou `export(doc_ezdxf)` + `_dataframe_to_ezdxf_doc`, `dataframe_to_geodataframe`

In [1]:
# --- Imports ---
import logging
import os
import shutil
import subprocess
from pathlib import Path

# --- Config ---
LIBREDWG_DIR = os.environ.get("LIBREDWG_DIR", r"C:\Users\mvm\libredwg")
CONVERT_TIMEOUT = 120  # secondes
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)


def _find_dwg2dxf() -> str | None:
    """Cherche dwg2dxf dans PATH ou LIBREDWG_DIR. Retourne le chemin ou None."""
    exe = "dwg2dxf.exe" if os.name == "nt" else "dwg2dxf"
    if path := shutil.which(exe):
        return path
    for sub in ("", "bin"):
        candidate = Path(LIBREDWG_DIR) / sub / exe
        if candidate.is_file():
            return str(candidate)
    return None


def convert_dwg_to_dxf(
    dwg_path: str,
    dxf_path: str,
    *,
    dwg2dxf_path: str | None = None,
    overwrite: bool = True,
    timeout: int = CONVERT_TIMEOUT,
) -> None:
    """Convertit un fichier DWG en DXF via LibreDWG (dwg2dxf)."""
    src = Path(dwg_path).resolve()
    dest = Path(dxf_path).resolve()

    if not src.is_file():
        raise FileNotFoundError(f"Fichier source introuvable : {src}")
    if src.stat().st_size == 0:
        raise ValueError(f"Fichier source vide : {src}")

    exe = dwg2dxf_path or _find_dwg2dxf()
    if not exe or not Path(exe).is_file():
        raise FileNotFoundError(
            "dwg2dxf introuvable. Installez LibreDWG : "
            "https://github.com/LibreDWG/libredwg/releases"
        )

    dest.parent.mkdir(parents=True, exist_ok=True)
    cmd = [exe, "-y" if overwrite else None, "-o", str(dest), str(src)]
    cmd = [c for c in cmd if c is not None]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"Timeout ({timeout}s) dépassé pour {src.name}")
    except OSError as e:
        raise RuntimeError(f"Impossible d'exécuter dwg2dxf : {e}") from e

    if result.returncode != 0:
        err = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"dwg2dxf a échoué (code {result.returncode}) : {err or 'aucune sortie'}")

    if not dest.is_file() or dest.stat().st_size == 0:
        raise RuntimeError(f"Fichier DXF non créé ou vide : {dest}")

# --- Conversion DWG → DXF ---
dwg_path = Path(r"C:\Users\mvm\Geolux_CV_Clone\01 Plans rez + sous-sol.dwg")
dxf_path = Path(r"C:\Users\mvm\OneDrive - Group Seco\Desktop\01 Plans rez + sous-sol_2.dxf")

convert_dwg_to_dxf(str(dwg_path), str(dxf_path), overwrite=True)
logger.info("Conversion terminée : %s → %s", dwg_path.name, dxf_path.name)

13:52:44 [INFO] Conversion terminée : 01 Plans rez + sous-sol.dwg → 01 Plans rez + sous-sol_2.dxf


In [2]:
# --- Imports ---
import ezdxf
import pandas as pd
from pathlib import Path

# --- Helpers ---
def _safe_utf8(s):
    """Échappe les caractères surrogate/invalides UTF-8."""
    if not isinstance(s, str):
        return s
    return s.encode("utf-8", errors="replace").decode("utf-8")


def _entity_to_row(path_name: str, entity) -> dict:
    """Transforme une entité DXF en dictionnaire pour DataFrame."""
    row = {
        "file_name": _safe_utf8(path_name),
        "entity_type": _safe_utf8(entity.dxftype()),
        "handle": getattr(entity.dxf, "handle", None),
    }
    for key, value in entity.dxfattribs().items():
        if value is None or isinstance(value, (int, float, bool)):
            row[key] = value
        else:
            row[key] = _safe_utf8(str(value)) if not isinstance(value, str) else _safe_utf8(value)
    return row


# --- Fichiers DXF à traiter ---
src = Path(dxf_path)
paths = [src] if src.is_file() else sorted(src.glob("*.dxf"), key=lambda p: p.name.lower())
if not paths:
    raise FileNotFoundError("Aucun fichier DXF. Exécutez d'abord la cellule de conversion.")

# --- Lecture et construction du DataFrame ---
rows, failed = [], []
for i, p in enumerate(paths, start=1):
    try:
        doc = ezdxf.readfile(str(p))
        count = 0
        for entity in doc.modelspace():
            rows.append(_entity_to_row(p.name, entity))
            count += 1
        print(f"[{i}/{len(paths)}] {p.name} — {count} entités")
    except Exception as e:
        failed.append((p.name, str(e)))
        print(f"[{i}/{len(paths)}] {p.name} — Échec: {e}")

dxf = pd.DataFrame(rows)
print(f"Terminé : {len(dxf)} entités, {len(failed)} échec(s)")

13:52:45 [INFO] creating ACAD_COLOR dictionary


[1/1] 01 Plans rez + sous-sol_2.dxf — 12093 entités
Terminé : 12093 entités, 0 échec(s)


In [3]:
dxf.head()

,file_name,entity_type,handle,owner,layer,linetype,lineweight,color,start,end,...,flow_direction,style,line_spacing_style,line_spacing_factor,major_axis,ratio,start_param,end_param,location,angle
0,01 Plans rez + sous-sol_2.dxf,LINE,60,1F,-1._-1. Sous-sol_2_0,Tirets,35,7,"(238378.5904506303, -793781.8124532328, 0.0)","(233345.9546988159, -793607.6114048628, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01 Plans rez + sous-sol_2.dxf,LINE,62,1F,-1._-1. Sous-sol_2_0,Tirets,35,7,"(249345.9560470913, -793824.1838479128, 0.0)","(249345.9661233813, -742079.8811148502, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01 Plans rez + sous-sol_2.dxf,LINE,63,1F,-1._-1. Sous-sol_2_0,Tirets,35,7,"(233345.9544737271, -742548.1154094319, 0.0)","(234094.6495531543, -742079.8809717703, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01 Plans rez + sous-sol_2.dxf,LINE,64,1F,-1._-1. Sous-sol_2_0,Tirets,35,7,"(233345.9546988159, -793607.6114048628, 0.0)","(227566.8927684401, -793407.6216325299, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01 Plans rez + sous-sol_2.dxf,LINE,65,1F,-1._-1. Sous-sol_2_0,Tirets,35,7,"(249345.9559206392, -793824.1835597546, 0.0)","(239186.4978771816, -793790.8827850878, 0.0)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
dxf["layer"].unique()

<ArrowStringArray>
[                                        '-1._-1. Sous-sol_2_0',
                                    '-1._-1. Sous-sol_2_limite',
                                  '-1._-1. Sous-sol_2_Hachures',
                              '-1._-1. Sous-sol_2_Sol _ Dalles',
                             '-1._-1. Sous-sol_2_MUR EXTERIEUR',
                           '-1._-1. Sous-sol_2_Portes Archicad',
                             '-1._-1. Sous-sol_2_MUR INTERIEUR',
                         '-1._-1. Sous-sol_2_Equip _ égouttage',
                             '-1._-1. Sous-sol_2_Poteaux béton',
                          '-1._-1. Sous-sol_2_Equip _ escalier',
                          '-1._-1. Sous-sol_2_Equip _ incendie',
                                 '-1._-1. Sous-sol_2_STABILITE',
                          '-1._-1. Sous-sol_2_Equip _ mobilier',
                               '-1._-1. Sous-sol_2_Descente EP',
                                    '-1._-1. Sous-sol_2_Lignes',
      

In [5]:
import pandas as pd

layers_geo = ["TERRASSES",
              "MUR PORTEUR", 
              "CLOISONS",
              "PORTES",
              "FENETRES",
              "A SUPPRIMER",
              "ESCALIERS",
              "COUPE",
              "TEXTE-LOT",
              "SURFACE-LOT",
              "LIMITE PARCELLAIRE",
              "TEXTE-LOT-NUMERO",
              "EXTERIEUR",
              "CADRE-CARTOUCHE",
              "PARKING",
              "HACHURES",
              "TEXTE",
              "HAUTEUR 1-2m",
              "Par défaut",
              "COTATIONS MUR",
              "COTATIONS"]
layers_to_keep = ["TERRASSES", "MUR PORTEUR", "CLOISONS", "PORTES", "FENETRES", "ESCALIERS", "COTATION"]
hachures_alternatives = [ "HACHURE", "Schraffur","hatching"]
cotations_alternatives = ["COTATION","quoting","zitieren"]
cotation_murs_alternatives = ["COTATION MUR"]
terrasses_alternatives = [
    
    "TERRASSE", "AUSSENBEREICH", "TERRASSENBEREICH",  # allemand
    "TERRASE","TERRACE"
]
murs_porteurs_alternatives = [
    "MUR", 
    "TRAG", "MAUER", "WAND",  # allemand
    "PORTANTE", "Räume","WALL"
]
cloisons_alternatives = [
    "CLOISON",
    "TRENNWAN", "INNENWAN",  # allemand
    "CLOISON", "PAROI"
]
portes_alternatives = [
    "PORTE", "PORTAIL",
    "TÜR","DOOR"
]
fenetres_alternatives = [
    "FENETRE", "OUVERTURE",
    "FENSTER","BAY","WINDOW"
]
escaliers_alternatives = [
    "ESCALIER", "STAIRS",
    "TREPPE", "HAUPTTREPPE", "STIEGE",  # allemand
]
coupes_alternatives = ["COUPE","CUT","schneiden"]
textes_alternatives = ["TEXT","TEXTEN"]
surfaces_alternatives = ["SURFACE","AREA"]
limites_parcellaires_alternatives = ["LIMITE","GRENZE","PARCELLE"]
parkings_alternatives = ["PARKING","PARKHAUS"]

alternatives = [terrasses_alternatives, 
                murs_porteurs_alternatives, 
                cloisons_alternatives, 
                portes_alternatives, 
                fenetres_alternatives, 
                escaliers_alternatives,
                hachures_alternatives,
                cotations_alternatives,
                cotation_murs_alternatives,
                coupes_alternatives,
                textes_alternatives,
                surfaces_alternatives,
                limites_parcellaires_alternatives,
                parkings_alternatives
                ]

                





In [6]:
def lower_array(array):
    """Met tous les éléments d'un tableau en minuscules."""
    return [str(value).lower() for value in array]


def normalize_layer_by_alternatives(column: pd.Series, alternatives: list, layers_geo: list) -> pd.Series:
    """
    Parse chaque cellule (str) de la colonne : si une alternative est présente,
    remplace par la valeur layers_geo associée.
    """
    # Association alternatives[i] → layers_geo à l'indice correspondant
    idx_map = [0, 1, 2, 3, 4, 6, 15, 20, 19, 7, 16, 9, 10, 14]  # indices dans layers_geo
    layers_lower = lower_array(layers_geo)
    alt_to_layer = {}
    for i, alt_list in enumerate(alternatives):
        target = layers_lower[idx_map[i]]
        for alt in alt_list:
            alt_to_layer[str(alt).lower().strip()] = target

    # Tri par longueur décroissante pour prioriser "cotation mur" avant "cotation"
    items_sorted = sorted(alt_to_layer.items(), key=lambda x: -len(x[0]))

    def _map_cell(val):
        if pd.isna(val):
            return val
        s = str(val).lower()
        for alt_key, layer_val in items_sorted:
            if alt_key in s:
                return layer_val
        return val

    return column.apply(_map_cell)


def rename_layer(df: pd.DataFrame, old_layer: str, new_layer: str, layer_col: str = "layer", inplace: bool = False) -> pd.DataFrame | None:
    """
    Renomme toutes les lignes dont le layer est old_layer vers new_layer.
    inplace=False : retourne une copie modifiée ; inplace=True : modifie df et retourne None.
    """
    out = df if inplace else df.copy()
    mask = out[layer_col] == old_layer
    out.loc[mask, layer_col] = new_layer
    return None if inplace else out


In [7]:
# --- Export : DataFrame / doc ezdxf → DXF ---
import ast
import ezdxf
import pandas as pd
from pathlib import Path
from ezdxf.addons.drawing import RenderContext, Frontend
from ezdxf.addons.drawing import dxf as dxf_backend


def _parse_point(val):
    """Parse (x,y,z) en tuple. Gère str, tuple ou NaN."""
    if pd.isna(val):
        return None
    if isinstance(val, (tuple, list)) and len(val) >= 2:
        return (float(val[0]), float(val[1]), float(val[2]) if len(val) > 2 else 0.0)
    s = str(val).strip()
    if not s or s.lower() == "nan":
        return None
    try:
        t = ast.literal_eval(s)
        return (float(t[0]), float(t[1]), float(t[2]) if len(t) > 2 else 0.0)
    except (ValueError, TypeError, SyntaxError):
        return None


# Linetypes standard (toujours présents) — les autres doivent être enregistrés
_STD_LINETYPES = {"ByBlock", "ByLayer", "Continuous"}

# Patterns de repli pour linetypes courants non standards
_LINETYPE_PATTERNS = {
    "tirets": [0.5, 0.25, -0.25],
    "tirets petits": [0.35, 0.15, -0.2],
    "tirets moyens": [0.8, 0.4, -0.4],
    "dashed": [0.5, 0.25, -0.25],
    "dashdot": [0.5, 0.25, -0.125, 0.0, -0.125],
}


def _dxfattribs_from_row(row: pd.Series, layer_col: str = "layer") -> dict:
    """Extrait layer, color, linetype, lineweight pour dxfattribs."""
    d = {}
    if layer_col in row and pd.notna(row[layer_col]):
        d["layer"] = str(row[layer_col])
    if "color" in row and pd.notna(row["color"]):
        d["color"] = int(row["color"])
    if "linetype" in row and pd.notna(row["linetype"]):
        d["linetype"] = str(row["linetype"])
    if "lineweight" in row and pd.notna(row["lineweight"]):
        d["lineweight"] = int(row["lineweight"])
    return d


def _ensure_linetype(doc, name: str) -> None:
    """Enregistre la linetype dans doc si elle n'existe pas (requis par AutoCAD)."""
    if not name or name in _STD_LINETYPES or name in doc.linetypes:
        return
    key = name.lower().strip()
    pattern = _LINETYPE_PATTERNS.get(key)
    if pattern is None:
        pattern = [0.5, 0.25, -0.25]  # fallback dashed
    try:
        doc.linetypes.add(name=name, pattern=pattern, description=name)
    except Exception:
        pass


def _dataframe_to_ezdxf_doc(
    df: pd.DataFrame,
    *,
    layer_col: str = "layer",
    entity_col: str = "entity_type",
) -> ezdxf.document.Drawing:
    """
    Construit un document ezdxf à partir d'un DataFrame DXF.
    Entités supportées : LINE, POINT, CIRCLE, ARC, LWPOLYLINE.
    Les layers sont explicitement enregistrés dans la table LAYER pour AutoCAD.
    """
    doc = ezdxf.new("R2010")
    msp = doc.modelspace()
    supported = {"LINE", "POINT", "CIRCLE", "ARC", "LWPOLYLINE"}

    # Enregistrer layers et linetypes dans les tables (requis par AutoCAD)
    layers, linetypes = set(), set()
    for _, row in df.iterrows():
        if str(row.get(entity_col, "")).upper() not in supported:
            continue
        attrs = _dxfattribs_from_row(row, layer_col)
        name = attrs.get("layer") or str(row.get(layer_col, "0"))
        if pd.notna(name):
            layers.add(str(name))
        lt = attrs.get("linetype")
        if lt and pd.notna(lt):
            linetypes.add(str(lt))
    for ln in layers:
        if ln and ln not in doc.layers:
            doc.layers.add(ln)
    for lt in linetypes:
        _ensure_linetype(doc, lt)

    for _, row in df.iterrows():
        etype = str(row.get(entity_col, "")).upper()
        if etype not in supported:
            continue
        attrs = _dxfattribs_from_row(row, layer_col)
        attrs.setdefault("layer", "0")
        if "linetype" in attrs and attrs["linetype"] not in doc.linetypes:
            del attrs["linetype"]  # éviter référence à une linetype non enregistrée

        if etype == "LINE":
            start, end = _parse_point(row.get("start")), _parse_point(row.get("end"))
            if start and end:
                msp.add_line(start, end, dxfattribs=attrs)
        elif etype == "POINT":
            loc = _parse_point(row.get("location"))
            if loc:
                msp.add_point(loc, dxfattribs=attrs)
        elif etype == "CIRCLE":
            center = _parse_point(row.get("center"))
            radius = row.get("radius")
            if center is not None and pd.notna(radius):
                msp.add_circle(center, float(radius), dxfattribs=attrs)
        elif etype == "ARC":
            center = _parse_point(row.get("center"))
            radius = row.get("radius")
            start_angle = row.get("start_angle", 0)
            end_angle = row.get("end_angle", 360)
            if center is not None and pd.notna(radius):
                msp.add_arc(center, float(radius), float(start_angle), float(end_angle), dxfattribs=attrs)
        elif etype == "LWPOLYLINE":
            points = row.get("points")
            if points is not None:
                if isinstance(points, str):
                    try:
                        points = ast.literal_eval(points)
                    except (ValueError, TypeError, SyntaxError):
                        continue
                pts = [(_parse_point(p) or (0, 0, 0))[:2] for p in (points if isinstance(points, list) else [points])]
                if pts:
                    msp.add_lwpolyline(pts, dxfattribs=attrs)
    return doc


def export(input_data, output_path: str | Path = "output_01.dxf"):
    """
    Exporte un DataFrame ou un document ezdxf vers un fichier DXF.
    - DataFrame : utilise _dataframe_to_ezdxf_doc
    - ezdxf doc : utilise le pipeline Frontend/Backend
    """
    if isinstance(input_data, pd.DataFrame):
        doc = _dataframe_to_ezdxf_doc(input_data)
        doc.saveas(output_path)
    else:
        doc = input_data
        export_doc = ezdxf.new()
        msp = doc.modelspace()
        context = RenderContext(doc)
        backend = dxf_backend.DXFBackend(export_doc.modelspace())
        frontend = Frontend(context, backend)
        frontend.draw_layout(msp)
        export_doc.saveas(output_path)


def dataframe_to_geodataframe(
    df: pd.DataFrame,
    *,
    entity_col: str = "entity_type",
    crs: str = "EPSG:2056",
) -> "geopandas.GeoDataFrame":
    """
    Convertit un DataFrame DXF en GeoDataFrame (optionnel, pour analyses spatiales).
    Nécessite : pip install geopandas shapely
    """
    import geopandas as gpd
    from shapely.geometry import Point, LineString

    def _pt(v):
        p = _parse_point(v)
        return Point(p[0], p[1]) if p else None

    geoms, indices = [], []
    for idx, row in df.iterrows():
        etype = str(row.get(entity_col, "")).upper()
        geom = None
        if etype == "LINE":
            start, end = _parse_point(row.get("start")), _parse_point(row.get("end"))
            if start and end:
                geom = LineString([(start[0], start[1]), (end[0], end[1])])
        elif etype == "POINT":
            geom = _pt(row.get("location"))
        if geom is not None:
            geoms.append(geom)
            indices.append(idx)

    if not geoms:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=crs)
    return gpd.GeoDataFrame(df.loc[indices].copy(), geometry=geoms, crs=crs)

In [8]:
# Applique la normalisation sur la colonne "layer"
dxf["layer"] = normalize_layer_by_alternatives(dxf["layer"], alternatives, layers_geo)


In [12]:
new_dxf = rename_layer(dxf, "-1._-1. Sous-sol_2_0", "test")
new_dxf["layer"].unique()

<ArrowStringArray>
[                                                        'test',
                                           'limite parcellaire',
                                                     'hachures',
                              '-1._-1. Sous-sol_2_Sol _ Dalles',
                                                  'mur porteur',
                                                       'portes',
                         '-1._-1. Sous-sol_2_Equip _ égouttage',
                             '-1._-1. Sous-sol_2_Poteaux béton',
                                                    'escaliers',
                          '-1._-1. Sous-sol_2_Equip _ incendie',
                                 '-1._-1. Sous-sol_2_STABILITE',
                          '-1._-1. Sous-sol_2_Equip _ mobilier',
                               '-1._-1. Sous-sol_2_Descente EP',
                                    '-1._-1. Sous-sol_2_Lignes',
                                                    'cotations',
      

In [10]:
export(new_dxf, "output_01.dxf")

13:52:45 [INFO] creating ACAD_COLOR dictionary
13:52:45 [INFO] creating ACAD_GROUP dictionary
13:52:45 [INFO] creating ACAD_LAYOUT dictionary
13:52:45 [INFO] creating ACAD_MATERIAL dictionary
13:52:45 [INFO] creating ACAD_MLEADERSTYLE dictionary
13:52:45 [INFO] creating ACAD_MLINESTYLE dictionary
13:52:45 [INFO] creating ACAD_PLOTSETTINGS dictionary
13:52:45 [INFO] creating ACAD_PLOTSTYLENAME dictionary
13:52:45 [INFO] creating ACAD_SCALELIST dictionary
13:52:45 [INFO] creating ACAD_TABLESTYLE dictionary
13:52:45 [INFO] creating ACAD_VISUALSTYLE dictionary
13:52:48 [INFO] did not write header var $INTERFEREOBJVS, value is None.
13:52:48 [INFO] did not write header var $INTERFEREVPVS, value is None.


In [11]:
print(doc.layers.name)

LAYER
